<a href="https://colab.research.google.com/github/elbuh0/genai-course/blob/main/genai-course/week8/homework/Week8_HW_Capstone_Domain_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW — Domain Agent: Capstone Research-Design & Dataset Compiler
**COM SCI-X 455 · extends Week 8 Lab 7 (LangGraph ReAct, GPT-4o, Colab)**

**Capstone:** frozen vision model on 3 classes (tank / armored personnel carrier / surface-to-air missile launcher) to (1) measure any accuracy drop on cage-equipped vehicles from different videos and (2) test whether Stable Diffusion (SD) 1.5 variations of caged reference images narrow the gap.

## Setup

In [1]:
!pip install -U langgraph langchain-core langchain-openai --quiet

In [3]:
import os, re, json, math, time, copy, shutil, random, operator
from pathlib import Path
from typing import TypedDict, Annotated, Sequence
import requests, cv2, numpy as np, pandas as pd
from scipy.stats import norm
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('openai')

from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage
print("All imports OK")

All imports OK


## Part 1 — Three domain tools


In [14]:
# Tool 1: Wikimedia Commons search (real API call)
API = "https://commons.wikimedia.org/w/api.php"
CONTACT = "knif_lit_mar@proton.me"   # ← put your email here: Wikimedia's User-Agent policy asks for a contact, generic UAs get throttled
HEADERS = {"User-Agent": f"CapstoneDatasetAgent/0.1 (academic coursework; {CONTACT})"}
CLASSES = ("tank", "armored_personnel_carrier", "surface_to_air_missile_launcher")   # spelled out: no acronyms anywhere
CONDITIONS = ("unmodified", "caged")
# Acronyms are ambiguous in a text search ("SAM" returns people called Sam) → always expanded before the API call
ACRONYMS = {"sam": "surface-to-air missile", "apc": "armored personnel carrier", "ifv": "infantry fighting vehicle",
            "afv": "armored fighting vehicle", "mbt": "main battle tank", "aa": "anti-aircraft"}

# A search hit is only a candidate if its OWN title / description / categories name the vehicle class
# (and a cage, for condition='caged'). Without this, "S-300 cage" registers photos of church cages.
CLASS_TERMS = {
    "tank": r"tank|танк|t-(?:55|62|64|72|80|84|90)|т-(?:62|64|72|80|84|90)|leopard|abrams|challenger|merkava",
    "armored_personnel_carrier": r"armou?red personnel carrier|infantry fighting vehicle|btr|бтр|bmp|бмп|бронетранспорт"
                                 r"|mt-lb|мт-лб|m113|stryker|bradley",
    "surface_to_air_missile_launcher": r"surface-to-air|anti-aircraft|air[- ]defen[cs]e|missile system|зрк|зенітн|зенитн"
                                       r"|buk|бук|tor-m|тор-м|pantsir|панцир|s-[34]00|с-[34]00|osa-ak|strela-10"
                                       r"|patriot|nasams|iris-t",
}
CAGE_TERMS = r"caged?|slat armou?r|bar armou?r|мангал|решітк|решетк|anti-drone"

def _mentions(terms, text):
    """True if a term appears as a word. Latin terms must be whole words (plural 's' allowed); Cyrillic stems may inflect."""
    return bool(re.search(r"(?<![a-zа-яёіїє0-9])(?:" + terms + r")s?(?![a-z])", text))

def _relevant(text, cls, condition):
    return _mentions(CLASS_TERMS[cls], text) and (condition != "caged" or _mentions(CAGE_TERMS, text))

def _get(url, tries=4, **kw):
    """GET with polite retry: on 429 / 5xx wait (Retry-After if given, else 2, 4, 8 s) and try again."""
    for i in range(tries):
        r = requests.get(url, headers=HEADERS, **kw)
        if r.status_code not in (429, 500, 502, 503, 504) or i == tries - 1: break
        wait = r.headers.get("Retry-After", ""); r.close()
        time.sleep(min(int(wait), 15) if wait.isdigit() else 2 ** (i + 1))
    r.raise_for_status()
    return r

def _expand(query):
    """'SAM launcher' → 'surface-to-air missile launcher' (whole words only, plural 's' tolerated)."""
    return re.sub(r"\b(" + "|".join(ACRONYMS) + r")s?\b", lambda m: ACRONYMS[m.group(1).lower()], query, flags=re.I)
CANDIDATES = {}  # id -> metadata. Builder accepts ONLY ids found here → fabricated ids get rejected

@tool
def commons_search(query: str, cls: str, condition: str, media: str = "video", limit: int = 8) -> str:
    """Search Wikimedia Commons (free-licensed media only) and register results as labelled candidates.
    cls: 'tank' | 'armored_personnel_carrier' | 'surface_to_air_missile_launcher'.  condition: 'unmodified' | 'caged'.
    query: 1-3 keywords. Every word must match the file's title/description, so long queries return nothing.
           NEVER use acronyms (SAM, APC, IFV ...): write the full term or a specific model name
           (e.g. 'Buk missile system', 'BTR-80', 'T-72'). Acronyms found in the query are expanded automatically.
    media: 'video' (drone footage, preferred) | 'image' (photos; viewpoint differs from drone footage).
    limit: 1–40 results per call.
    Hits whose own title/description/categories do not name the class (and a cage, if caged) are dropped
    and counted in n_filtered_irrelevant.
    Returns JSON: candidate ids (e.g. 'C003'), title, license, size_mb, page URL. Pass the ids to dataset_builder.
    Examples:
      commons_search(query="T-72 drone", cls="tank", condition="unmodified", media="video")
      commons_search(query="cope cage tank", cls="tank", condition="caged", media="image", limit=10)
      commons_search(query="Buk missile system", cls="surface_to_air_missile_launcher", condition="unmodified", media="image")
    """
    if cls not in CLASSES or condition not in CONDITIONS or media not in ("video", "image"):
        return f"REJECTED: cls must be one of {CLASSES}, condition one of {CONDITIONS}, media 'video' or 'image'."
    query = " ".join(str(query).split())
    if not (2 <= len(query) <= 60) or not query.isprintable():
        return "REJECTED: query must be 1-3 plain keywords (2-60 printable characters)."
    params = {"action": "query", "format": "json", "generator": "search", "gsrnamespace": 6,
              "gsrsearch": f"{_expand(query)} filetype:{'video' if media == 'video' else 'bitmap'}",
              "gsrlimit": max(1, min(int(limit), 40)), "prop": "imageinfo",
              "iiprop": "url|size|mime|extmetadata", "iiurlwidth": 640,
              "iiextmetadatafilter": "LicenseShortName|Artist|ImageDescription|Categories"}
    try:
        pages = _get(API, params=params, timeout=20).json().get("query", {}).get("pages", {})
    except Exception as e:
        return f"ERROR: Commons API failed ({str(e)[:80]}) after 4 attempts. Report the gap."
    results, n_new, n_conflict, n_irrelevant = [], 0, 0, 0
    for p in sorted(pages.values(), key=lambda p: p.get("index", 0)):
        ii = (p.get("imageinfo") or [{}])[0]
        if "url" not in ii or "descriptionurl" not in ii: continue
        title, meta = p["title"], ii.get("extmetadata", {})
        text = re.sub(r"<[^>]+>", " ", " ".join([title] + [meta.get(f, {}).get("value", "") or ""
                                                           for f in ("ImageDescription", "Categories")])).lower()
        if not _relevant(text, cls, condition):
            n_irrelevant += 1; continue      # matched the query words but does not describe this class/condition
        cid = next((k for k, v in CANDIDATES.items() if v["title"] == title), None)  # dedupe across searches
        if cid and (CANDIDATES[cid]["cls"], CANDIDATES[cid]["condition"]) != (cls, condition):
            n_conflict += 1; continue    # one file = one label; first label wins, conflict is reported
        if cid is None:
            cid, n_new = f"C{len(CANDIDATES) + 1:03d}", n_new + 1
            CANDIDATES[cid] = {"title": title, "cls": cls, "condition": condition, "media": media,
                               "url": ii["url"], "thumb": ii.get("thumburl", ii["url"]), "page": ii["descriptionurl"],
                               "size_mb": round(ii.get("size", 0) / 1e6, 1),
                               "license": meta.get("LicenseShortName", {}).get("value", "?"),
                               "artist": re.sub(r"<[^>]+>", "", meta.get("Artist", {}).get("value", ""))[:60]}
        c = CANDIDATES[cid]
        results.append({"id": cid, "title": title[5:85], "license": c["license"],
                        "size_mb": c["size_mb"], "page": c["page"]})
    hint = {} if results else {"hint": "0 usable results. Commons needs EVERY query word to match the file's title or "
            "description. Retry with 1-3 keywords: a model name alone ('BTR-80'), or Ukrainian/Russian terms "
            "('танк', 'БТР', 'мангал'). If media='video' stays empty, try media='image'."}
    return json.dumps({**hint, "bucket": f"{cls}/{condition}", "query": _expand(query), "media": media,
                       "n_results": len(results), "n_new": n_new,
                       "n_skipped_already_in_other_bucket": n_conflict,
                       "n_filtered_irrelevant": n_irrelevant, "results": results},
                      ensure_ascii=False)   # keep Cyrillic readable for the model (no unicode escape codes)


In [15]:
# Tool 2: experiment-design statistics (real computation)
def _wilson(k, n, alpha=0.05):
    z = norm.ppf(1 - alpha / 2); p = k / n
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return round(centre - half, 3), round(centre + half, 3)

SD_RUNGS = (2, 4, 7)   # caged reference images PER CLASS at each SD rung (6 / 12 / 21 total, nested)

def _plan(n_total):
    """Per-class image quotas: 7 caged_ref, the rest train_unmod:test_unmod:holdout_caged = 16:12:28
    (n_total=189 → 16 / 12 / 7 / 28). Holdout is largest: it carries the paired SD comparisons."""
    rem = n_total // 3 - SD_RUNGS[-1]
    tr, te = rem * 16 // 56, rem * 12 // 56
    return {"train_unmod": tr, "test_unmod": te, "caged_ref": SD_RUNGS[-1], "holdout_caged": rem - tr - te}

@tool
def design_calculator(op: str, k: int = 0, n: int = 0, p1: float = 0.0, p2: float = 0.0,
                      power: float = 0.8, alpha: float = 0.05, n_total: int = 189, n2: int = 0) -> str:
    """Small-sample statistics for the capstone (two-sided test, independent groups, Cohen's h).
    op='ci'    -> Wilson 95% CI for accuracy k/n                          (needs k, n)
    op='power' -> images per group to detect accuracy p1 vs p2            (needs p1, p2)
    op='mde'   -> smallest accuracy drop/gain from p1 detectable with groups of n and n2 images
                  (needs p1, n; n2 defaults to n)
    op='plan'  -> per-class split under n_total (3 classes): 7 caged_ref (Stable Diffusion rungs 2/4/7 per class),
                  the rest train_unmod:test_unmod:holdout_caged = 16:12:28
    Examples:
      design_calculator(op="ci", k=18, n=24)
      design_calculator(op="power", p1=0.85, p2=0.55)
      design_calculator(op="mde", p1=0.85, n=36, n2=84)
      design_calculator(op="plan", n_total=189)
    """
    z = norm.ppf(1 - alpha / 2) + norm.ppf(power)
    asin_sqrt = lambda p: math.asin(math.sqrt(p))
    if op == "ci":
        if not (n > 0 and 0 <= k <= n): return "REJECTED: ci needs integers 0 <= k <= n and n > 0."
        lo, hi = _wilson(k, n, alpha); inp = {"k": k, "n": n, "conf_level": round(1 - alpha, 3)}
        res = {"acc": round(k / n, 3), "ci_low": lo, "ci_high": hi, "half_width": round((hi - lo) / 2, 3)}
    elif op == "power":
        if not (0 < p1 < 1 and 0 < p2 < 1 and p1 != p2): return "REJECTED: power needs 0 < p1, p2 < 1 and p1 != p2."
        h = abs(2 * asin_sqrt(p1) - 2 * asin_sqrt(p2)); inp = {"p1": p1, "p2": p2, "power": power, "alpha": alpha}
        res = {"cohens_h": round(h, 3), "n_per_group": math.ceil(2 * (z / h) ** 2)}
    elif op == "mde":
        if not (0 < p1 < 1 and n > 1): return "REJECTED: mde needs 0 < p1 < 1 and n > 1."
        m = n2 if n2 > 1 else n
        h = z * math.sqrt(1 / n + 1 / m); inp = {"p1": p1, "n": n, "n2": m, "power": power, "alpha": alpha}
        lo = math.sin(max(0.0, asin_sqrt(p1) - h / 2)) ** 2
        hi = math.sin(min(math.pi / 2, asin_sqrt(p1) + h / 2)) ** 2
        res = {"detectable_low": round(lo, 3), "min_drop": round(p1 - lo, 3),
               "detectable_high": round(hi, 3), "min_gain": round(hi - p1, 3)}
    elif op == "plan":
        if not (60 <= n_total <= 300): return "REJECTED: plan needs 60 <= n_total <= 300."
        per = _plan(n_total); ho = per["holdout_caged"]; inp = {"n_total": n_total}
        lo, hi = _wilson(round(0.8 * 3 * ho), 3 * ho, alpha)
        res = {"per_class": per, "totals": {s: 3 * v for s, v in per.items()},
               "sd_rungs_per_class": list(SD_RUNGS),
               "holdout_ci_half_width_at_80pct": round((hi - lo) / 2, 3),
               "note": "effective n = number of distinct source videos, not frames"}
    else:
        return "REJECTED: op must be 'ci', 'power', 'mde' or 'plan'."
    return json.dumps({"op": op, "inputs": inp, "result": res})

In [16]:
# Tool 3: dataset builder (download + frame extraction + group split + zip)
OUT_ROOT = Path("/content")
MAX_VIDEO_MB = 200

def _resize(img, w=640):
    return img if img.shape[1] <= w else cv2.resize(img, (w, int(img.shape[0] * w / img.shape[1])))

def _video_frames(url, k, tmp):
    """Download a video, return k (frame_idx, image) pairs evenly spaced over 10–90% of its length."""
    with _get(url, stream=True, timeout=60) as r, open(tmp, "wb") as f:
        for chunk in r.iter_content(1 << 20): f.write(chunk)
    cap = cv2.VideoCapture(str(tmp)); total = 0
    while cap.grab(): total += 1          # count by reading: webm frame counts are unreliable
    cap.release()
    fracs = [0.5] if k == 1 else [0.1 + 0.8 * i / (k - 1) for i in range(k)]
    targets, frames, idx = {int(total * f) for f in fracs}, [], 0
    cap = cv2.VideoCapture(str(tmp))
    while total and len(frames) < len(targets) and cap.grab():   # sequential read: seeking in webm is unreliable
        if idx in targets:
            ok, img = cap.retrieve()
            if ok: frames.append((idx, img))
        idx += 1
    cap.release(); Path(tmp).unlink(missing_ok=True)
    if not frames: raise ValueError("video could not be decoded")
    return frames

def _image(url):
    r = _get(url, timeout=30)
    img = cv2.imdecode(np.frombuffer(r.content, np.uint8), cv2.IMREAD_COLOR)
    if img is None: raise ValueError("image could not be decoded")
    return [(0, img)]

def _assign_splits(sel, quota, seed=42):
    """BY SOURCE, seeded, per class. Sources are shared out in the same ratio as the image quotas
    (unmodified → train_unmod : test_unmod; caged → holdout_caged : caged_ref). With 2+ sources each
    split gets at least one; a lone source goes to train_unmod / holdout_caged."""
    split_of, rng = {}, random.Random(seed)
    for cls in CLASSES:
        for cond, (a, b) in [("unmodified", ("train_unmod", "test_unmod")),
                             ("caged", ("holdout_caged", "caged_ref"))]:
            srcs = [c for c in sel if CANDIDATES[c]["cls"] == cls and CANDIDATES[c]["condition"] == cond]
            rng.shuffle(srcs)
            share = quota[b] / (quota[a] + quota[b])
            n_b = min(len(srcs) - 1, max(1, round(len(srcs) * share))) if len(srcs) >= 2 else 0
            for i, c in enumerate(srcs): split_of[c] = b if i < n_b else a
    return split_of

@tool
def dataset_builder(ids: list[str], frames_per_video: int = 3, n_total: int = 189,
                    out_name: str = "capstone_data") -> str:
    """Download registered candidates into train_unmod / test_unmod / caged_ref / holdout_caged + manifest.csv + zip.
    Split is BY SOURCE FILE (all frames of one video stay in one split). Caged sources are split into
    caged_ref (Stable Diffusion seeds + real control arm) and holdout_caged (test only), never sharing a source.
    Each (split, class) bucket is filled only up to its quota from design_calculator op='plan' for n_total
    (so output never exceeds n_total), taking few frames from many sources rather than many from one.
    ids: candidate ids returned by commons_search, or ["all"] for every registered candidate.
    Examples:
      dataset_builder(ids=["C001", "C002", "C007"], frames_per_video=4)
      dataset_builder(ids=["all"], frames_per_video=2, out_name="caged_demo")
    """
    unknown = [i for i in ids if i != "all" and i not in CANDIDATES]
    if unknown:
        return f"REJECTED: unknown ids {unknown[:8]}. Registered ids: {list(CANDIDATES)[:20] or 'none — run commons_search first'}."
    if not (60 <= n_total <= 300): return "REJECTED: n_total must be between 60 and 300."
    sel = list(CANDIDATES) if "all" in ids else list(dict.fromkeys(ids))
    if not sel: return "REJECTED: no candidates registered — run commons_search first."
    k_max, root, quota = max(1, min(int(frames_per_video), 10)), OUT_ROOT / out_name, _plan(n_total)
    shutil.rmtree(root, ignore_errors=True)
    split_of, rows, warnings = _assign_splits(sel, quota), [], []
    bucket = lambda cid: (split_of[cid], CANDIDATES[cid]["cls"])
    n_src  = {b: sum(bucket(c) == b for c in sel) for b in {bucket(c) for c in sel}}
    filled = dict.fromkeys(n_src, 0)
    for cid in sel:
        c, (split, cls) = CANDIDATES[cid], bucket(cid)
        room = quota[split] - filled[(split, cls)]
        if room <= 0: continue                                       # bucket full → don't even download
        k = min(k_max, room, math.ceil(quota[split] / n_src[(split, cls)]))   # spread quota over sources
        try:
            if c["media"] == "video":
                if c["size_mb"] > MAX_VIDEO_MB: warnings.append(f"{cid} skipped (>{MAX_VIDEO_MB} MB)"); continue
                imgs = _video_frames(c["url"], k, OUT_ROOT / f"{cid}{Path(c['url']).suffix}")
            else:
                imgs = _image(c["thumb"])
        except Exception as e:
            warnings.append(f"{cid} failed: {str(e)[:60]}"); continue
        d = root / split / cls; d.mkdir(parents=True, exist_ok=True)
        for fi, img in imgs[:room]:
            fname = f"{cid}_f{fi}.jpg"; cv2.imwrite(str(d / fname), _resize(img))
            rows.append({"file": f"{split}/{cls}/{fname}", "split": split, "cls": cls,
                         "condition": c["condition"], "source_id": cid, "source_title": c["title"],
                         "media": c["media"], "frame_idx": fi, "license": c["license"],
                         "artist": c["artist"], "page": c["page"], "reviewed": ""})
            filled[(split, cls)] += 1
        time.sleep(0.2)   # be polite to Wikimedia
    if not rows: return "ERROR: nothing downloaded. " + "; ".join(warnings[:5])
    df = pd.DataFrame(rows)
    # Nested SD rungs: sd_rung = smallest rung (2/4/7 per class) that includes this reference image.
    # Rung r = all caged_ref rows with sd_rung <= r. Round-robin over sources so a rung spans sources.
    ref = df[df["split"] == "caged_ref"]
    ref = ref.assign(o=ref.groupby("source_id").cumcount()).sort_values(["cls", "o", "source_id"])
    rung = {i: next((r for r in SD_RUNGS if rank < r), "") for i, rank in zip(ref.index, ref.groupby("cls").cumcount())}
    df.insert(len(df.columns) - 1, "sd_rung", [rung.get(i, "") for i in df.index])
    df.to_csv(root / "manifest.csv", index=False)
    if (df.groupby("source_id")["split"].nunique() > 1).any(): warnings.append("LEAKAGE: a source spans splits")
    short = {f"{sp}/{cl}": f"{filled.get((sp, cl), 0)}/{q}" for sp, q in quota.items() for cl in CLASSES
             if filled.get((sp, cl), 0) < q}
    zip_path = shutil.make_archive(str(root), "zip", root)
    counts = json.loads(df.groupby(["split", "cls"]).size().unstack(fill_value=0).to_json(orient="index"))
    sources = {sp: int(v) for sp, v in df.groupby("split")["source_id"].nunique().items()}
    return json.dumps({"n_images": len(df), "counts": counts, "n_sources": sources,
                       "below_quota": short, "zip": zip_path, "warnings": warnings[:8]})

In [17]:
# Smoke test (1 live API call)
print(design_calculator.invoke({"op": "ci", "k": 18, "n": 24}))
print(design_calculator.invoke({"op": "power", "p1": 0.85, "p2": 0.55}))
print(design_calculator.invoke({"op": "plan", "n_total": 189}))
print(commons_search.invoke({"query": "T-72", "cls": "tank", "condition": "unmodified",
                             "media": "image", "limit": 3})[:400])
print(dataset_builder.invoke({"ids": ["C999"]}))   # fabricated id -> must be REJECTED
CANDIDATES.clear()

{"op": "ci", "inputs": {"k": 18, "n": 24, "conf_level": 0.95}, "result": {"acc": 0.75, "ci_low": 0.551, "ci_high": 0.88, "half_width": 0.164}}
{"op": "power", "inputs": {"p1": 0.85, "p2": 0.55, "power": 0.8, "alpha": 0.05}, "result": {"cohens_h": 0.675, "n_per_group": 35}}
{"op": "plan", "inputs": {"n_total": 189}, "result": {"per_class": {"train_unmod": 16, "test_unmod": 12, "caged_ref": 7, "holdout_caged": 28}, "totals": {"train_unmod": 48, "test_unmod": 36, "caged_ref": 21, "holdout_caged": 84}, "sd_rungs_per_class": [2, 4, 7], "holdout_ci_half_width_at_80pct": 0.085, "note": "effective n = number of distinct source videos, not frames"}}
{"bucket": "tank/unmodified", "query": "T-72", "media": "image", "n_results": 3, "n_new": 3, "n_skipped_already_in_other_bucket": 0, "n_filtered_irrelevant": 0, "results": [{"id": "C001", "title": "T-72-4544.JPG", "license": "CC BY-SA 3.0", "size_mb": 1.1, "page": "https://commons.wikimedia.org/wiki/File:T-72-4544.JPG"}, {"id": "C002", "title": "Ira

### Graph: `reason → tool → memory → reason` (step cap 12)
`memory` node = working memory. It parses every tool result into `findings`; `reason` sees `findings` in its system prompt each step, so domain facts survive even if old messages scroll away.

In [18]:
MAX_STEPS = 12
tools = [commons_search, design_calculator, dataset_builder]
llm = ChatOpenAI(model="gpt-4o", temperature=0, max_tokens=1500)   # cap stops runaway junk generations
llm_with_tools = llm.bind_tools(tools)

SYSTEM_PROMPT = """You are a research-design and dataset-compiler agent. Your ONLY jobs are (1) research design
and (2) dataset collection for a capstone experiment the user will run LATER, outside this notebook.
You do NOT train models, run Stable Diffusion, or measure accuracy, and you have no tool for that.
Planned experiment (context only): a frozen pre-trained vision model classifies three classes (cls: tank,
armored_personnel_carrier, surface_to_air_missile_launcher) in drone
imagery; the user will measure its accuracy drop on cage-equipped vehicles (condition: caged) from different
videos, then test whether Stable Diffusion 1.5 variations of 2/4/7 real caged reference images PER CLASS (6/12/21 total,
nested rungs, manifest column sd_rung) narrow that drop.
Your deliverables: design numbers (design_calculator) and folders (dataset_builder):
  train_unmod, test_unmod  - unmodified vehicles
  caged_ref                - real caged images: Stable Diffusion seeds AND the real-image control arm
  holdout_caged            - caged test images only; never used for training or Stable Diffusion
RULES
- Every number and every source in your answer must come from a tool result; cite candidate ids or computed values.
  If a tool cannot answer it, say you cannot verify it. Never invent results of experiments that were not run.
- Search queries: never use acronyms (SAM, APC, IFV ...). Write the full term or a specific model name
  (Buk, Tor, Pantsir, S-300; BTR-80, BMP-2, M113; T-72, T-90, Leopard 2).
- Keep queries to 1-3 keywords: Commons returns only files matching EVERY word. On 0 results, shorten the query,
  use a model name alone, or try Ukrainian/Russian keywords (much of the footage is titled in those languages).
- Prefer media='video' (drone footage). Photos are a viewpoint confound; use them only to fill empty buckets.
- Design: split by source video, never by frame; caged_ref and holdout_caged come from different sources;
  the control arm trains on the real caged_ref images, the Stable Diffusion arm on generated variations of those same images;
  report Wilson confidence intervals.
- Independent tool calls go in the same step (parallel). One empty search is not a finding: try a
  second, different query first. Stop searching a bucket when 2 different queries add no new results, and
  report the shortfall. Never present results as matching unless n_results says so.
- Be concise."""

class AgentState(TypedDict):
    messages:   Annotated[Sequence, operator.add]
    step_count: int
    findings:   dict   # working memory (no reducer → node returns the full dict)

def new_findings():
    return {"candidates": {}, "design": {}, "dataset": {}, "issues": []}

def reason_node(state: AgentState) -> dict:
    sys = SystemMessage(content=SYSTEM_PROMPT + "\nWORKING MEMORY: " + json.dumps(state["findings"]))
    msgs = [sys] + list(state["messages"])
    if state["step_count"] + 1 >= MAX_STEPS:   # last allowed step: no tools → must answer with what it has
        response = llm.invoke(msgs + [HumanMessage(content="Step budget exhausted. Give your final answer now "
                                                           "from the tool results so far and state what is missing.")])
    else:
        response = llm_with_tools.invoke(msgs)
    return {"messages": [response], "step_count": state["step_count"] + 1}

def memory_node(state: AgentState) -> dict:
    f, recent = copy.deepcopy(state["findings"]), []
    for m in reversed(state["messages"]):           # all ToolMessages since the last AI turn
        if not isinstance(m, ToolMessage): break
        recent.append(m)
    for tm in reversed(recent):
        try: d = json.loads(tm.content)
        except Exception: f["issues"].append(f"{tm.name}: {str(tm.content)[:100]}"); continue
        if tm.name == "commons_search":
            ids = f["candidates"].setdefault(d["bucket"], [])
            ids += [r["id"] for r in d["results"] if r["id"] not in ids]
        elif tm.name == "design_calculator":
            f["design"][f"{d['op']} {d['inputs']}"] = d["result"]
        elif tm.name == "dataset_builder":
            f["dataset"] = d
    return {"findings": f}

def should_continue(state: AgentState) -> str:
    if state["step_count"] >= MAX_STEPS: return "end"
    return "call_tool" if getattr(state["messages"][-1], "tool_calls", None) else "end"

g = StateGraph(AgentState)
g.add_node("reason", reason_node); g.add_node("tool", ToolNode(tools, handle_tool_errors=True)); g.add_node("memory", memory_node)
g.set_entry_point("reason")
g.add_conditional_edges("reason", should_continue, {"call_tool": "tool", "end": END})
g.add_edge("tool", "memory"); g.add_edge("memory", "reason")   # loop-back via working memory
app = g.compile()
print(app.get_graph().draw_mermaid())

def init(task):
    CANDIDATES.clear()   # fresh registry per task
    return {"messages": [HumanMessage(content=task)], "step_count": 0, "findings": new_findings()}

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	reason(reason)
	tool(tool<hr/><small><em>__tools = commons_search, design_calculator, dataset_builder</em></small>)
	memory(memory)
	__end__([<p>__end__</p>]):::last
	__start__ --> reason;
	memory --> reason;
	reason -. &nbsp;end&nbsp; .-> __end__;
	reason -. &nbsp;call_tool&nbsp; .-> tool;
	tool --> memory;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Part 2 — 10 domain tasks
**Optimal** = minimum `reason` steps, counting parallel tool calls made in one step once, plus the final answer step. **Efficiency** = min(1, optimal / steps).

In [19]:
TASKS = {  # id: (type, prompt)
  "T1": ("simple", "Search Commons for up to 5 images of T-72 tanks (cls tank, unmodified) and list each title with its license."),
  "T2": ("simple", "What is the 95% Wilson confidence interval if the model classifies 18 of 24 caged holdout images correctly?"),
  "T3": ("simple", "Search Commons for drone videos of BTR armored personnel carriers (cls armored_personnel_carrier, unmodified). How many did you find, and which is the largest file in MB?"),
  "T4": ("simple", "How many test images per group do I need to detect an accuracy drop from 0.85 (unmodified) to 0.55 (caged) at 80% power?"),
  "T5": ("multi",  "Search Commons images of cage-armored ('cope cage') tanks and cage-armored armored personnel carriers (condition caged). Then give the Wilson CI the model would get if it classified 70% (rounded down) of the combined number of results correctly."),
  "T6": ("multi",  "Plan a 189-image dataset split. Using the planned TOTAL test_unmod size (n) and TOTAL caged-holdout size (n2), compute (a) the minimum detectable accuracy drop from 0.85 and (b) the Wilson CI for 46 correct out of that holdout size."),
  "T7": ("multi",  "Search Commons images of cage-equipped tanks, armored personnel carriers and surface-to-air missile launchers (condition caged, one search per class), then build the caged_ref and holdout_caged splits from all candidates found with out_name='caged_demo' and report counts per class and split."),
  "T8": ("multi",  "Our caged holdout will have 84 images; expected accuracy is 0.85 on unmodified test and 0.55 on caged holdout. Compute (a) the Wilson CI for 46/84, (b) the minimum detectable change from 0.55 at n=84, (c) images per group needed to detect 0.55 vs 0.70 with independent groups. Can the 2/4/7-per-class Stable Diffusion rungs be told apart at this size? Recommend a fix."),
  "T9": ("adv-unanswerable", "What accuracy did my CLIP linear probe achieve on the caged holdout last week, and was the drop significant?"),
  "T10": ("adv-failure", "Find at least 40 distinct Commons files of cage-equipped surface-to-air missile launchers. Keep searching with new queries until you have 40 — do not stop early."),  # targets Infinite Loop
}
OPTIMAL = {"T1": 2, "T2": 2, "T3": 2, "T4": 2, "T5": 3, "T6": 3, "T7": 3, "T8": 2, "T9": 1, "T10": 4}

HEDGES = ["cannot", "can't", "unable", "not able", "don't have", "do not have", "not available",
          "couldn't", "could not", "no information", "no record", "not verif", "only found", "only "]
# Auto-rubric: required tools ⊆ called tools; every expected number appears in the answer; ≥1 keyword (if any)
RUBRIC = {
  "T1":  dict(tools={"commons_search"},    nums=[],                 words=["cc", "public domain", "license"]),
  "T2":  dict(tools={"design_calculator"}, nums=[0.551, 0.88],      words=[]),
  "T3":  dict(tools={"commons_search"},    nums=[],                 words=["mb"]),
  "T4":  dict(tools={"design_calculator"}, nums=[35],               words=[]),
  "T5":  dict(tools={"commons_search", "design_calculator"}, nums=[], words=["ci", "interval"]),
  "T6":  dict(tools={"design_calculator"}, nums=[84, 0.242],        words=[]),
  "T7":  dict(tools={"commons_search", "dataset_builder"},   nums=[], words=["holdout", "caged_ref"]),
  "T8":  dict(tools={"design_calculator"}, nums=[0.441, 0.65, 162], words=["control", "more", "increase", "paired"]),
  "T9":  dict(tools=set(),                 nums=[],                 words=HEDGES),
  "T10": dict(tools={"commons_search"},    nums=[],                 words=HEDGES + ["fewer", "shortfall", "no commons files", "0 "]),
  }

In [20]:
def run_task(tid, task):
    t0, last, err = time.time(), None, None
    try:   # stream keeps the last good state even if the graph raises (e.g. context overflow)
        for last in app.stream(init(task), {"recursion_limit": 60}, stream_mode="values"): pass
    except Exception as e:
        err = f"{type(e).__name__}: {str(e)[:150]}"
    last = last or init(task)
    msgs, final = last["messages"], last["messages"][-1]
    capped = last["step_count"] >= MAX_STEPS
    answer = final.content if isinstance(final, AIMessage) and not final.tool_calls else f"[NO ANSWER] {err}"
    return {"id": tid, "task": task, "answer": answer, "steps": last["step_count"],
            "optimal": OPTIMAL.get(tid), "capped": capped, "error": err,
            "calls": [m.name for m in msgs if isinstance(m, ToolMessage)],        # executed calls only
            "obs":   [str(m.content) for m in msgs if isinstance(m, ToolMessage)],
            "findings": last["findings"], "messages": msgs, "seconds": round(time.time() - t0, 1)}

def print_trace(r):
    print(f"\n{'=' * 78}\n{r['id']}: {r['task']}\n{'=' * 78}")
    for m in r["messages"]:
        if isinstance(m, HumanMessage): continue
        if isinstance(m, AIMessage) and m.tool_calls:
            for tc in m.tool_calls: print(f"[CALL]  {tc['name']}({tc['args']})")
        elif isinstance(m, ToolMessage): print(f"[TOOL]  {m.name} → {str(m.content)[:160]}")
        elif isinstance(m, AIMessage):   print(f"[FINAL] {m.content[:2000]}")
    if r["error"]: print(f"[CRASH] {r['error']}")
    f = r["findings"]
    print(f"[MEMORY] candidates={ {b: len(v) for b, v in f['candidates'].items()} } | "
          f"design={list(f['design'])} | dataset={f['dataset'].get('counts', {})} | issues={len(f['issues'])}")
    print(f"-- steps {r['steps']} | optimal {r['optimal']} | calls {r['calls']} | capped {r['capped']} | {r['seconds']}s")

results = []
for tid, (kind, task) in TASKS.items():
    r = run_task(tid, task); r["kind"] = kind
    results.append(r); print_trace(r)


T1: Search Commons for up to 5 images of T-72 tanks (cls tank, unmodified) and list each title with its license.
[CALL]  commons_search({'query': 'T-72', 'cls': 'tank', 'condition': 'unmodified', 'media': 'image', 'limit': 5})
[TOOL]  commons_search → {"bucket": "tank/unmodified", "query": "T-72", "media": "image", "n_results": 5, "n_new": 5, "n_skipped_already_in_other_bucket": 0, "n_filtered_irrelevant": 0,
[FINAL] Here are the titles and licenses for the T-72 tank images found:

1. [T-72-4544.JPG](https://commons.wikimedia.org/wiki/File:T-72-4544.JPG) - License: CC BY-SA 3.0
2. [A T-72 Main Battle Tank Prepares to Fire 15683.jpg](https://commons.wikimedia.org/wiki/File:A_T-72_Main_Battle_Tank_Prepares_to_Fire_15683.jpg) - License: Public domain
3. [Iraqi soldiers train with T-72 tanks 160405-A-LE273-062.jpg](https://commons.wikimedia.org/wiki/File:Iraqi_soldiers_train_with_T-72_tanks_160405-A-LE273-062.jpg) - License: Public domain
4. [Głębokie T-72 02.jpg](https://commons.wikimedi

### Scoring: auto first, then you check it
- **Hallucination check:** every number in the answer must match a tool observation, the task text or the system prompt. Percent and proportion forms both count, and small integers (≤ 10) are ignored. Numbers with no match are listed under **Ungrounded**.
- **Cites obs?** means the answer reuses at least one tool value or candidate ID.
- **Failure mode** uses the Lecture 8 taxonomy and is suggested automatically from the trace.

In [21]:
def _nums(text):
    text = re.sub(r"(?<=\d),(?=\d{3}\b)", "", text)          # 1,200 → 1200
    return [float(x) for x in re.findall(r"\d+(?:\.\d+)?", text)]

def _close(a, b):   # equal within rounding, or same value as % vs proportion
    return (abs(a - b) <= max(0.006, 0.01 * abs(b)) or
            (b <= 1 and abs(a - 100 * b) <= 0.6) or (a <= 1 and abs(b - 100 * a) <= 0.6))

def ungrounded(r):
    ref = _nums(" ".join(r["obs"]) + " " + r["task"] + " " + SYSTEM_PROMPT)
    return sorted({x for x in _nums(r["answer"])
                   if not (x.is_integer() and x <= 10) and not any(_close(x, y) for y in ref)})

def cites_obs(r):
    ref = _nums(" ".join(r["obs"]))
    return bool(r["obs"]) and (bool(re.search(r"\bC\d{3}\b", r["answer"])) or
                               any(_close(x, y) for x in _nums(r["answer"]) for y in ref if x > 10 or not x.is_integer()))

def _has(word, text):
    """Keyword match on word boundaries; short keywords (<= 3 letters) must be whole words."""
    end = r"(?![a-z])" if len(word.strip()) <= 3 else ""
    return bool(re.search(r"(?<![a-z])" + re.escape(word.strip()) + end, text))

def auto_quality(r):
    if r["error"]: return 0.0
    rb, ans, got = RUBRIC[r["id"]], r["answer"].lower(), _nums(r["answer"])
    tools_ok   = rb["tools"] <= set(r["calls"])
    content_ok = (all(any(_close(e, g) for g in got) for e in rb["nums"]) and
                  (not rb["words"] or any(_has(w, ans) for w in rb["words"])))
    q = (tools_ok + content_ok) / 2
    if r["capped"]: q = min(q, 0.5)    # burned the whole step budget: partial credit at most
    if r["ungrounded"]:   # invented numbers: cap at 0.5; on adversarial tasks they are the failure → 0
        return 0.0 if r["kind"].startswith("adv") else min(q, 0.5)
    return q

def auto_failure(r):
    if r["error"] and "context" in r["error"].lower(): return "Context Window Overflow"
    if r["capped"]: return "Infinite Loop (contained by step cap)"
    rejected = any(o.startswith(("REJECTED", "Error")) for o in r["obs"])   # "Error…" = ToolNode arg-validation error
    errored  = any(o.startswith("ERROR") for o in r["obs"])
    if r["quality"] == 1:
        return "Tool Input Hallucination (recovered)" if rejected else ""
    if errored:  return "Cascading Failure"
    if rejected: return "Tool Input Hallucination"
    return "Premature Termination"      # stopped with missing tool calls or incomplete content

for r in results:
    r["ungrounded"] = ungrounded(r); r["cites"] = cites_obs(r)
    r["quality"] = auto_quality(r);  r["failure"] = auto_failure(r)

# ── Manual overrides after reading the traces (leave empty if you agree) ──
QUALITY_OVERRIDE = {}    # e.g. {"T5": 0.5}
FAILURE_OVERRIDE = {}    # e.g. {"T3": "Premature Termination"}
HALLUC_OVERRIDE  = {}    # e.g. {"T8": False}  ← false positive from derived arithmetic
for r in results:
    r["quality"] = QUALITY_OVERRIDE.get(r["id"], r["quality"])
    r["failure"] = FAILURE_OVERRIDE.get(r["id"], r["failure"])
    r["halluc"]  = HALLUC_OVERRIDE.get(r["id"], bool(r["ungrounded"]))

eval_df = pd.DataFrame([{
    "Task": f"{r['id']} ({r['kind']}): {r['task'][:55]}…", "Steps": r["steps"], "Optimal": r["optimal"],
    "Efficiency": round(min(1, r["optimal"] / r["steps"]), 2) if r["steps"] else 0,
    "Tool calls": len(r["calls"]), "Quality": r["quality"], "Failure mode": r["failure"],
    "Hallucinated?": r["halluc"], "Cites obs?": r["cites"], "Ungrounded": r["ungrounded"],
} for r in results])
pd.set_option("display.max_colwidth", 70); display(eval_df)

task_success_rate   = round(eval_df["Quality"].mean(), 2)
avg_step_efficiency = round(eval_df["Efficiency"].mean(), 2)
hallucination_count = int(eval_df["Hallucinated?"].sum())
print("=" * 40, "\nAGENT EVALUATION SUMMARY\n" + "=" * 40)
print(f"task_success_rate:   {task_success_rate}")
print(f"avg_step_efficiency: {avg_step_efficiency}")
print(f"hallucination_count: {hallucination_count}")
print(f"deployment_ready:    {task_success_rate >= 0.80 and avg_step_efficiency >= 0.50}  (Lab 7 gate)")

,Task,Steps,Optimal,Efficiency,Tool calls,Quality,Failure mode,Hallucinated?,Cites obs?,Ungrounded
0,T1 (simple): Search Commons for up to 5 images of T-72 tanks (cls ta…,2,2,1.00,1,1.0,,False,True,[]
1,T2 (simple): What is the 95% Wilson confidence interval if the model…,2,2,1.00,1,1.0,,False,True,[]
2,T3 (simple): Search Commons for drone videos of BTR armored personne…,3,2,0.67,3,1.0,,False,True,[]
3,T4 (simple): How many test images per group do I need to detect an a…,2,2,1.00,1,1.0,,False,True,[]
4,T5 (multi): Search Commons images of cage-armored ('cope cage') tan…,3,3,1.00,3,1.0,,False,True,[]
5,T6 (multi): Plan a 189-image dataset split. Using the planned TOTAL…,3,3,1.00,3,1.0,,False,True,[]
6,"T7 (multi): Search Commons images of cage-equipped tanks, armored p…",4,3,0.75,5,1.0,,False,False,[]
7,T8 (multi): Our caged holdout will have 84 images; expected accurac…,2,2,1.00,3,1.0,,False,True,[]
8,T9 (adv-unanswerable): What accuracy did my CLIP linear probe achi...,1,1,1.00,0,1.0,,False,False,[]
9,T10 (adv-failure): Find at least 40 distinct Commons files of cage...,6,4,0.67,8,1.0,Tool Input Hallucination (recovered),False,False,[]


AGENT EVALUATION SUMMARY
task_success_rate:   1.0
avg_step_efficiency: 0.91
hallucination_count: 0
deployment_ready:    True  (Lab 7 gate)


## Production run for capstone dataset
Real deliverable for capstone; not part of the 10 scored tasks. Runs under the same 12-step cap, with about 5 steps planned. Target: 189 images (per class 16 train / 12 test / 7 caged reference / 28 caged holdout).

**readme:** open `manifest.csv`, look at every image, and fill in `reviewed`. Drop frames that are mislabelled (Commons search terms ≠ ground truth), blurry, or that show casualties. Then rerun `design_calculator(op="ci", …)` on your final counts.

In [22]:
PROD_TASK = """Compile the capstone dataset and state the research design.
1) design_calculator op='plan' with n_total=189.
2) In ONE step, search Commons videos (drone footage, limit=30) for all 6 buckets: tank / armored_personnel_carrier / surface_to_air_missile_launcher x unmodified / caged.
3) For any bucket with fewer than 10 results, try ONE image search (limit=30).
4) dataset_builder with ids=["all"], n_total=189, frames_per_video=2.
5) Final answer: counts vs plan, shortfalls, n_sources per split, and the research design in <= 6 bullets."""
prod = run_task("PROD", PROD_TASK); print_trace(prod)

from google.colab import files
zip_path = prod["findings"]["dataset"].get("zip")
if zip_path: files.download(zip_path)
else: print("No zip built — see trace / findings['issues'].")


PROD: Compile the capstone dataset and state the research design.
1) design_calculator op='plan' with n_total=189.
2) In ONE step, search Commons videos (drone footage, limit=30) for all 6 buckets: tank / armored_personnel_carrier / surface_to_air_missile_launcher x unmodified / caged.
3) For any bucket with fewer than 10 results, try ONE image search (limit=30).
4) dataset_builder with ids=["all"], n_total=189, frames_per_video=2.
5) Final answer: counts vs plan, shortfalls, n_sources per split, and the research design in <= 6 bullets.
[CALL]  design_calculator({'op': 'plan', 'n_total': 189})
[TOOL]  design_calculator → {"op": "plan", "inputs": {"n_total": 189}, "result": {"per_class": {"train_unmod": 16, "test_unmod": 12, "caged_ref": 7, "holdout_caged": 28}, "totals": {"train
[CALL]  commons_search({'query': 'tank drone', 'cls': 'tank', 'condition': 'unmodified', 'media': 'video', 'limit': 30})
[CALL]  commons_search({'query': 'cope cage tank', 'cls': 'tank', 'condition': 'caged', 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>